<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/07_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 · Memory: AGENTS.md and memory files

"Give the agent memory" sounds like one feature. It is **three different mechanisms**, with three
different owners and three different update cycles — and mixing them up is why so many agents
either forget everything or remember the wrong things forever.

| Tier | Who writes it | Changes how often | Lives where |
|---|---|---|---|
| **System prompt** | an engineer | at deploy time | your code |
| **`AGENTS.md`** | anyone on the team | any time, no deploy | the backend |
| **`/memories/`** | **the agent itself** | continuously | the store |

**New in this lesson:** `memory=[...]`, store-backed `/memories/`, per-user namespaces

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq --progress-bar off \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-07-memory"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. Tier 2 — `AGENTS.md`

Tier 1 you already know: the system prompt. It ships with your code, it is identical for every
user, and changing it needs an engineer and a deployment. Right for what the agent **is**; wrong
for anything that varies by team, customer, or week.

An `AGENTS.md` file is read from the backend at startup and folded into the system prompt. It is
the knob you can hand to someone who does not write Python.

In [ ]:
!mkdir -p agent_home/memory

In [ ]:
%%writefile agent_home/memory/AGENTS.md
# House style

- Address the customer by first name only, never "Dear Sir/Madam".
- Never promise a delivery date. Say "typically 3-5 business days".
- Every reply ends with the ticket reference on its own line, formatted: `Ref: T-####`
- If a refund is involved, always state the policy line you relied on.
- British spelling.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

styled_agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a customer support agent for an office furniture retailer.",
    backend=FilesystemBackend(root_dir="agent_home"),
    memory=["/memory/AGENTS.md"],
)
styled_agent

In [ ]:
from langsmith_studio_nb import start_studio

start_studio("styled_agent")

Example prompt:
> Draft a reply to Avery for ticket T-1: their standing desk arrived with a cracked leg.

No code told it to sign off with a ref, or to use British spelling.

Now **edit the file below**, re-run it, re-run the agent cell, and send the same prompt again.
The behaviour changes with no Python touched — which is the whole point of this tier.

In [ ]:
%%writefile agent_home/memory/AGENTS.md
# House style

- Address the customer formally: "Dear Avery".
- Always give an explicit next step with a named owner.
- Do not include ticket references in the body; the ticketing system adds them.
- Keep replies under 60 words.
- American spelling.

Both tiers end up in the same system prompt, so why not just put house style in
`system_prompt`? Because they have **different owners and different release cycles**. A support
lead can edit `AGENTS.md` at 4pm on a Friday because the tone was wrong — no pull request, no
deploy, no engineer.

The test: **would a non-engineer reasonably want to change this without asking you?** If yes, it
belongs in `AGENTS.md`.

---

## 2. Tier 3 — memory the agent writes itself

The first two tiers are things *you* tell the agent. This one is what the agent learns and
records on its own, so a later conversation can use it.

There is no special machinery. The agent already has filesystem tools from lesson 02; you route
`/memories/` to a store so what it writes outlives the thread, and use the system prompt to say
when to write and when to look.

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend

learning_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are a helpful assistant for a support team.\n"
        "At the START of every conversation, list /memories/ and read what is there.\n"
        "When you learn a durable preference or fact about how this person works, "
        "record it under /memories/ so future conversations can use it."
    ),
    backend=CompositeBackend(
        default=StateBackend(),
        routes={"/memories/": StoreBackend(namespace=lambda rt: ("workshop", "memories"))},
    ),
)
learning_agent

In [ ]:
start_studio("learning_agent")

Example prompts:
> For future reference: I always want refund amounts shown in both USD and GBP, and I hate bullet points — write in short paragraphs.

> Order 1042 was refunded $429. Write me a one-line summary.

Send the first prompt, then **start a new thread** in Studio and send the second. The preference
carries over, even though nothing from the first conversation is in the second's messages.

### Recall is not the model remembering

Look at where the preference appears: **before the first model call**, because the agent listed
`/memories/` and read the file. The model is stateless and has no idea a previous conversation
happened.

So recall is **retrieval plus prompt assembly**. That reframing tells you where to debug: what
was written, what was retrieved, or what was assembled — three concrete, inspectable places, none
of them inside the model.

---

## 3. Namespaces: keeping users apart

The `namespace` callable receives the runtime, so memory can be scoped per user, per team, per
tenant. Get this wrong and you have built a data leak.

In [ ]:
from dataclasses import dataclass


@dataclass
class Context:
    user_id: str


def per_user(rt) -> tuple[str, ...]:
    # Scope every memory read and write to the current user.
    return ("users", rt.context.user_id, "memories")


scoped_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are a helpful assistant. At the start of every conversation, list /memories/ "
        "and read what is there. Record durable user preferences under /memories/."
    ),
    backend=CompositeBackend(
        default=StateBackend(),
        routes={"/memories/": StoreBackend(namespace=per_user)},
    ),
    context_schema=Context,
)
scoped_agent

In [ ]:
start_studio("scoped_agent")

Studio lets you set the run **context** in the config panel. Set `user_id` to `avery`, tell the
agent something, then change it to `jordan` and ask about it.

Example prompts:

> Remember: my department code is ENG-4471 and I approve refunds up to $500.

> What is my department code?

With `user_id=avery` it answers. With `user_id=jordan` it does not — the namespace keeps them in
separate keys.

---

## 4. How agent-written memory goes wrong

Three failure modes, all of which appear only after weeks in production:

**1. Hoarding.** Everything looks worth remembering, so `/memories/` grows without bound. It is
loaded into the prompt on every call, so cost per request climbs forever and quality drops as
signal thins out. *Fix:* tell the agent what qualifies, and prune on a schedule.

**2. Contradiction.** In March the user prefers bullet points; in June they do not. Both notes
are in the store, there is no timestamp to break the tie, and the agent behaves inconsistently.
*Fix:* instruct it to **update** existing memory files rather than only appending.

**3. Leakage.** The namespace is wrong, or absent, and one customer's preferences surface in
another's conversation. This is the one that becomes an incident report.

There is a fourth, subtler one: **the agent records something wrong and then trusts it forever.**
Memory has no error correction unless you build some.

---

## 📌 Key takeaways

- "Memory" is three mechanisms with three owners: system prompt (engineers), `AGENTS.md` (the team), `/memories/` (the agent).
- `AGENTS.md` is the tier a non-engineer can change without a deploy — that is its whole point.
- Recall is **retrieval plus prompt assembly**, not the model remembering. That tells you where to debug.
- Agent-written memory needs a namespace, and cross-user isolation must be tested, not assumed.
- Memory that only appends will eventually contradict itself; instruct the agent to update.
- Everything in `/memories/` is paid for on every model call, so hoarding is a cost bug and a quality bug.

---

## ➡️ Next

**[08 · Skills: progressive disclosure of procedures](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/08_skills.ipynb)**

`AGENTS.md` is always loaded. But an agent with a dozen procedures cannot afford to carry all of
them at once — which is what **skills** solve.